In [1]:
#@title Cell 1 - Install
%pip -q install ensemble-boxes pycocotools pandas numpy

In [2]:
#@title Cell 2 - Mount Drive and configure isolated paths

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from typing import Any, Dict, List, Sequence
from collections import defaultdict
from itertools import product
from contextlib import redirect_stdout
from io import StringIO
import json
import hashlib
import re

import numpy as np
import pandas as pd

V3_TAG = "multiseed_wbf_results_ready_v3_complete"
YOLOV10N_TAG = "multiseed_wbf_results_ready_v3_extension_yolov10n_v1"
OUTPUT_TAG = "multiseed_wbf_results_ready_v3_extension_final_combos"

PROTOCOLS = ["time_based", "source_based"]
SEEDS = [42, 123, 2026]

PAIRS = {
    "rtdetr_l_yolov10n": ["rtdetr_l", "yolov10n"],
    "yolo11m_yolov10n": ["yolo11m", "yolov10n"],
}

IOU_GRID = [0.45, 0.50, 0.55]
SKIP_GRID = [0.001, 0.01]
WEIGHT_GRID = [
    [1.00, 1.00],
    [1.25, 1.00],
    [1.00, 1.25],
]
CONF_TYPE = "avg"
MAX_DETECTIONS = 300

# This allows the notebook to run while YOLOv10n is still finishing.
# Completed protocol-seed cases are fused; incomplete cases are logged.
SKIP_MISSING_INPUTS = True

ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/au-fish"),
    Path(
        "/content/drive/.shortcut-targets-by-id/"
        "1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish"
    ),
]

def resolve_project_root() -> Path:
    for candidate in ROOT_CANDIDATES:
        if (candidate / "dataset_v2").is_dir():
            return candidate.resolve()

    for root in [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/.shortcut-targets-by-id"),
    ]:
        if not root.exists():
            continue

        for dataset_root in root.rglob("dataset_v2"):
            if dataset_root.is_dir():
                return dataset_root.parent.resolve()

    raise FileNotFoundError("Folder AU-FISH tidak ditemukan.")

PROJECT_ROOT = resolve_project_root()
DATASET_ROOT = PROJECT_ROOT / "dataset_v2"

V3_ROOT = DATASET_ROOT / "experiments" / V3_TAG
V3_CANONICAL = V3_ROOT / "canonical_data"
V3_PREDICTIONS = V3_ROOT / "predictions"

YOLOV10N_ROOT = DATASET_ROOT / "experiments" / YOLOV10N_TAG
YOLOV10N_PREDICTIONS = YOLOV10N_ROOT / "predictions"

OUTPUT_ROOT = DATASET_ROOT / "experiments" / OUTPUT_TAG
NEW_WBF_ROOT = OUTPUT_ROOT / "new_wbf"
REPORTS_ROOT = OUTPUT_ROOT / "reports"
MANIFESTS_ROOT = OUTPUT_ROOT / "manifests"

for folder in [NEW_WBF_ROOT, REPORTS_ROOT, MANIFESTS_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

def load_json(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def atomic_json(value: Any, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")

    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(
            value,
            handle,
            ensure_ascii=False,
            indent=2,
            default=str,
        )

    temporary.replace(path)

def save_csv(dataframe: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(path, index=False)

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()

print("V3:", V3_ROOT)
print("YOLOv10n:", YOLOV10N_ROOT)
print("Output:", OUTPUT_ROOT)

Mounted at /content/drive
V3: /content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish/dataset_v2/experiments/multiseed_wbf_results_ready_v3_complete
YOLOv10n: /content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish/dataset_v2/experiments/multiseed_wbf_results_ready_v3_extension_yolov10n_v1
Output: /content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish/dataset_v2/experiments/multiseed_wbf_results_ready_v3_extension_final_combos


In [3]:
#@title Cell 3 - WBF and COCO evaluation functions

from ensemble_boxes import weighted_boxes_fusion
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def image_sizes(gt_path: Path) -> Dict[int, Sequence[float]]:
    data = load_json(gt_path)

    return {
        int(image["id"]): (
            float(image["width"]),
            float(image["height"]),
        )
        for image in data["images"]
    }

def group_by_image(predictions: List[Dict[str, Any]]):
    grouped = defaultdict(list)

    for prediction in predictions:
        grouped[
            int(prediction["image_id"])
        ].append(prediction)

    return grouped

def fuse_predictions(
    predictions_a: List[Dict[str, Any]],
    predictions_b: List[Dict[str, Any]],
    gt_path: Path,
    iou_thr: float,
    skip_box_thr: float,
    weights: Sequence[float],
):
    sizes = image_sizes(gt_path)
    grouped_a = group_by_image(predictions_a)
    grouped_b = group_by_image(predictions_b)
    output = []

    for image_id in sorted(sizes):
        image_width, image_height = sizes[image_id]
        boxes_list = []
        scores_list = []
        labels_list = []

        for grouped in [grouped_a, grouped_b]:
            boxes = []
            scores = []
            labels = []

            for row in grouped.get(image_id, []):
                x, y, width, height = [
                    float(value)
                    for value in row["bbox"]
                ]

                x1 = max(0.0, min(1.0, x / image_width))
                y1 = max(0.0, min(1.0, y / image_height))
                x2 = max(
                    0.0,
                    min(
                        1.0,
                        (x + width) / image_width,
                    ),
                )
                y2 = max(
                    0.0,
                    min(
                        1.0,
                        (y + height) / image_height,
                    ),
                )

                if x2 <= x1 or y2 <= y1:
                    continue

                boxes.append([x1, y1, x2, y2])
                scores.append(float(row["score"]))
                labels.append(int(row["category_id"]))

            boxes_list.append(boxes)
            scores_list.append(scores)
            labels_list.append(labels)

        if not boxes_list[0] and not boxes_list[1]:
            continue

        boxes, scores, labels = weighted_boxes_fusion(
            boxes_list,
            scores_list,
            labels_list,
            weights=list(weights),
            iou_thr=float(iou_thr),
            skip_box_thr=float(skip_box_thr),
            conf_type=CONF_TYPE,
        )

        order = np.argsort(
            -np.asarray(scores)
        )[:MAX_DETECTIONS]

        for index in order:
            x1, y1, x2, y2 = boxes[index]

            output.append({
                "image_id": int(image_id),
                "category_id": int(labels[index]),
                "bbox": [
                    float(x1 * image_width),
                    float(y1 * image_height),
                    float((x2 - x1) * image_width),
                    float((y2 - y1) * image_height),
                ],
                "score": float(scores[index]),
            })

    return output

def per_class_ap(evaluator: COCOeval):
    precision = evaluator.eval.get("precision")

    if precision is None:
        return {}

    output = {}

    for category_index, category_id in enumerate(
        evaluator.params.catIds
    ):
        values = precision[
            :,
            :,
            category_index,
            0,
            -1,
        ]

        values = values[values > -1]

        output[int(category_id)] = (
            float(values.mean())
            if values.size
            else float("nan")
        )

    return output

def evaluate_coco_predictions(
    gt_path: Path,
    predictions: List[Dict[str, Any]],
    temporary_name: str,
):
    temporary_path = Path("/content") / temporary_name
    atomic_json(predictions, temporary_path)

    coco_gt = COCO(str(gt_path))

    if not predictions:
        temporary_path.unlink(missing_ok=True)

        return {
            "map50_95": 0.0,
            "map50": 0.0,
            "map75": 0.0,
            "coral_ap": 0.0,
            "fish_ap": 0.0,
            "worst_class_ap": 0.0,
            "detections": 0,
        }

    coco_dt = coco_gt.loadRes(str(temporary_path))
    evaluator = COCOeval(coco_gt, coco_dt, "bbox")
    evaluator.params.imgIds = sorted(coco_gt.getImgIds())
    evaluator.params.catIds = sorted(coco_gt.getCatIds())
    evaluator.params.maxDets = [1, 10, 100]

    buffer = StringIO()

    with redirect_stdout(buffer):
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    class_aps = per_class_ap(evaluator)
    coral_ap = float(class_aps.get(1, float("nan")))
    fish_ap = float(class_aps.get(2, float("nan")))

    temporary_path.unlink(missing_ok=True)

    return {
        "map50_95": float(evaluator.stats[0]),
        "map50": float(evaluator.stats[1]),
        "map75": float(evaluator.stats[2]),
        "coral_ap": coral_ap,
        "fish_ap": fish_ap,
        "worst_class_ap": float(
            np.nanmin([coral_ap, fish_ap])
        ),
        "detections": len(predictions),
    }

def selection_objective(metrics):
    return (
        float(metrics["map50_95"]),
        float(metrics["worst_class_ap"]),
        float(metrics["map50"]),
    )

def prediction_path(
    protocol: str,
    model: str,
    seed: int,
    split: str,
) -> Path:
    if model == "yolov10n":
        root = YOLOV10N_PREDICTIONS
    else:
        root = V3_PREDICTIONS

    return (
        root
        / protocol
        / model
        / f"seed_{seed}"
        / f"{split}.json"
    )

In [4]:
#@title Cell 4 - Validation tuning, test fusion, gains, and reports

gain_rows = []
tuning_rows = []
pending_rows = []

for protocol in PROTOCOLS:
    validation_gt = (
        V3_CANONICAL
        / protocol
        / "val"
        / "annotations.json"
    )

    test_gt = (
        V3_CANONICAL
        / protocol
        / "test"
        / "annotations.json"
    )

    for pair_name, members in PAIRS.items():
        member_a, member_b = members

        for seed in SEEDS:
            inputs = {
                "member_a_val": prediction_path(
                    protocol,
                    member_a,
                    seed,
                    "val",
                ),
                "member_b_val": prediction_path(
                    protocol,
                    member_b,
                    seed,
                    "val",
                ),
                "member_a_test": prediction_path(
                    protocol,
                    member_a,
                    seed,
                    "test",
                ),
                "member_b_test": prediction_path(
                    protocol,
                    member_b,
                    seed,
                    "test",
                ),
                "validation_gt": validation_gt,
                "test_gt": test_gt,
            }

            missing = [
                str(path)
                for path in inputs.values()
                if not path.exists()
            ]

            if missing:
                pending_rows.append({
                    "protocol": protocol,
                    "pair": pair_name,
                    "seed": seed,
                    "status": "PENDING_INPUTS",
                    "missing_paths": " | ".join(missing),
                })

                if SKIP_MISSING_INPUTS:
                    print(
                        "PENDING:",
                        protocol,
                        pair_name,
                        seed,
                    )
                    continue

                raise FileNotFoundError(
                    "\n".join(missing)
                )

            member_a_val = load_json(
                inputs["member_a_val"]
            )
            member_b_val = load_json(
                inputs["member_b_val"]
            )
            member_a_test = load_json(
                inputs["member_a_test"]
            )
            member_b_test = load_json(
                inputs["member_b_test"]
            )

            best = None

            for iou_thr, skip_thr, weights in product(
                IOU_GRID,
                SKIP_GRID,
                WEIGHT_GRID,
            ):
                validation_fused = fuse_predictions(
                    member_a_val,
                    member_b_val,
                    validation_gt,
                    iou_thr,
                    skip_thr,
                    weights,
                )

                validation_metrics = (
                    evaluate_coco_predictions(
                        validation_gt,
                        validation_fused,
                        (
                            f"new_wbf_val_"
                            f"{protocol}_"
                            f"{pair_name}_"
                            f"{seed}_"
                            f"{iou_thr}_"
                            f"{skip_thr}_"
                            f"{weights[0]}_"
                            f"{weights[1]}.json"
                        ),
                    )
                )

                tuning_row = {
                    "protocol": protocol,
                    "pair": pair_name,
                    "seed": seed,
                    "iou_thr": iou_thr,
                    "skip_box_thr": skip_thr,
                    "conf_type": CONF_TYPE,
                    "weight_a": weights[0],
                    "weight_b": weights[1],
                    **validation_metrics,
                }

                tuning_rows.append(
                    tuning_row
                )

                if (
                    best is None
                    or selection_objective(
                        tuning_row
                    )
                    > selection_objective(
                        best
                    )
                ):
                    best = tuning_row

            if best is None:
                raise RuntimeError(
                    "No valid WBF setting."
                )

            case_root = (
                NEW_WBF_ROOT
                / protocol
                / pair_name
                / f"seed_{seed}"
            )

            prediction_root = (
                case_root
                / "predictions"
            )

            result_root = (
                case_root
                / "results"
            )

            manifest_root = (
                case_root
                / "manifests"
            )

            for folder in [
                prediction_root,
                result_root,
                manifest_root,
            ]:
                folder.mkdir(
                    parents=True,
                    exist_ok=True,
                )

            best_parameters = {
                "selected_on": "validation",
                "selection_objective": [
                    "mAP50-95",
                    "worst-class AP",
                    "mAP50",
                ],
                "protocol": protocol,
                "pair": pair_name,
                "model_order": members,
                "seed": seed,
                "iou_thr": best["iou_thr"],
                "skip_box_thr": best[
                    "skip_box_thr"
                ],
                "conf_type": CONF_TYPE,
                "weights": [
                    best["weight_a"],
                    best["weight_b"],
                ],
            }

            parameter_path = (
                result_root
                / "best_wbf_parameters.json"
            )

            atomic_json(
                best_parameters,
                parameter_path,
            )

            test_fused = fuse_predictions(
                member_a_test,
                member_b_test,
                test_gt,
                best_parameters["iou_thr"],
                best_parameters[
                    "skip_box_thr"
                ],
                best_parameters["weights"],
            )

            test_prediction_path = (
                prediction_root
                / "test_wbf_fused.json"
            )

            atomic_json(
                test_fused,
                test_prediction_path,
            )

            member_a_metrics = (
                evaluate_coco_predictions(
                    test_gt,
                    member_a_test,
                    (
                        f"member_a_"
                        f"{protocol}_"
                        f"{pair_name}_"
                        f"{seed}.json"
                    ),
                )
            )

            member_b_metrics = (
                evaluate_coco_predictions(
                    test_gt,
                    member_b_test,
                    (
                        f"member_b_"
                        f"{protocol}_"
                        f"{pair_name}_"
                        f"{seed}.json"
                    ),
                )
            )

            fused_metrics = (
                evaluate_coco_predictions(
                    test_gt,
                    test_fused,
                    (
                        f"new_wbf_test_"
                        f"{protocol}_"
                        f"{pair_name}_"
                        f"{seed}.json"
                    ),
                )
            )

            best_member_map = max(
                member_a_metrics["map50_95"],
                member_b_metrics["map50_95"],
            )

            gain_row = {
                "protocol": protocol,
                "pair": pair_name,
                "member_a": member_a,
                "member_b": member_b,
                "seed": seed,
                "selected_iou_thr": (
                    best_parameters["iou_thr"]
                ),
                "selected_skip_box_thr": (
                    best_parameters[
                        "skip_box_thr"
                    ]
                ),
                "selected_weight_a": (
                    best_parameters[
                        "weights"
                    ][0]
                ),
                "selected_weight_b": (
                    best_parameters[
                        "weights"
                    ][1]
                ),
                "member_a_map50_95": (
                    member_a_metrics[
                        "map50_95"
                    ]
                ),
                "member_b_map50_95": (
                    member_b_metrics[
                        "map50_95"
                    ]
                ),
                "best_member_map50_95": (
                    best_member_map
                ),
                "wbf_map50_95": (
                    fused_metrics[
                        "map50_95"
                    ]
                ),
                "gain_over_best_member": (
                    fused_metrics[
                        "map50_95"
                    ]
                    - best_member_map
                ),
                "wbf_map50": (
                    fused_metrics["map50"]
                ),
                "wbf_map75": (
                    fused_metrics["map75"]
                ),
                "wbf_coral_ap": (
                    fused_metrics[
                        "coral_ap"
                    ]
                ),
                "wbf_fish_ap": (
                    fused_metrics[
                        "fish_ap"
                    ]
                ),
                "wbf_worst_class_ap": (
                    fused_metrics[
                        "worst_class_ap"
                    ]
                ),
                "parameter_path": str(
                    parameter_path
                ),
                "test_prediction_path": str(
                    test_prediction_path
                ),
            }

            gain_rows.append(gain_row)

            atomic_json(
                {
                    "inputs": {
                        key: str(value)
                        for key, value
                        in inputs.items()
                    },
                    "input_sha256": {
                        key: sha256_file(value)
                        for key, value
                        in inputs.items()
                    },
                    "best_parameters": (
                        best_parameters
                    ),
                    "test_prediction_sha256": (
                        sha256_file(
                            test_prediction_path
                        )
                    ),
                },
                manifest_root
                / "artifact_manifest.json",
            )

gain_df = pd.DataFrame(gain_rows)
tuning_df = pd.DataFrame(tuning_rows)
pending_df = pd.DataFrame(pending_rows)

save_csv(
    gain_df,
    REPORTS_ROOT
    / "new_wbf_gains.csv",
)

save_csv(
    tuning_df,
    REPORTS_ROOT
    / "new_wbf_validation_tuning.csv",
)

save_csv(
    pending_df,
    REPORTS_ROOT
    / "new_wbf_pending.csv",
)

summary_rows = []

if not gain_df.empty:
    for (
        protocol,
        pair,
    ), group in gain_df.groupby(
        ["protocol", "pair"]
    ):
        values = group[
            "gain_over_best_member"
        ].to_numpy(
            dtype=float
        )

        summary_rows.append({
            "protocol": protocol,
            "pair": pair,
            "n_seeds": len(group),
            "wbf_map50_95_mean": float(
                group[
                    "wbf_map50_95"
                ].mean()
            ),
            "gain_over_best_member_mean": (
                float(values.mean())
            ),
            "gain_over_best_member_sd": (
                float(
                    np.std(
                        values,
                        ddof=1,
                    )
                )
                if len(values) > 1
                else float("nan")
            ),
            "positive_gain_seed_count": int(
                (values > 0).sum()
            ),
        })

summary_df = pd.DataFrame(
    summary_rows
)

save_csv(
    summary_df,
    REPORTS_ROOT
    / "new_wbf_summary.csv",
)

completion = {
    "expected_cases": (
        len(PROTOCOLS)
        * len(PAIRS)
        * len(SEEDS)
    ),
    "completed_cases": len(gain_df),
    "pending_cases": len(pending_df),
    "output_report": str(
        REPORTS_ROOT
        / "new_wbf_gains.csv"
    ),
    "test_was_used_for_tuning": False,
    "ready_when_all_inputs_exist": bool(
        len(pending_df) == 0
    ),
}

atomic_json(
    completion,
    MANIFESTS_ROOT
    / "new_wbf_completion.json",
)

display(gain_df)
display(summary_df)
display(pending_df)
print(json.dumps(completion, indent=2))

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.31s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.30s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.30s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.27s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.27s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.27s)
creating index...
index created!
loading annotati

,protocol,pair,member_a,member_b,seed,selected_iou_thr,selected_skip_box_thr,selected_weight_a,selected_weight_b,member_a_map50_95,...,best_member_map50_95,wbf_map50_95,gain_over_best_member,wbf_map50,wbf_map75,wbf_coral_ap,wbf_fish_ap,wbf_worst_class_ap,parameter_path,test_prediction_path
0,time_based,rtdetr_l_yolov10n,rtdetr_l,yolov10n,42,0.55,0.010,1.00,1.25,0.436590,...,0.436590,0.335989,-0.100601,0.541870,0.356153,0.403823,0.268155,0.268155,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...
1,time_based,rtdetr_l_yolov10n,rtdetr_l,yolov10n,123,0.55,0.010,1.00,1.25,0.453966,...,0.453966,0.393061,-0.060905,0.597524,0.417619,0.498854,0.287268,0.287268,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...
2,time_based,rtdetr_l_yolov10n,rtdetr_l,yolov10n,2026,0.55,0.010,1.00,1.25,0.437297,...,0.437297,0.376931,-0.060365,0.589140,0.386176,0.483996,0.269867,0.269867,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...
3,time_based,yolo11m_yolov10n,yolo11m,yolov10n,42,0.55,0.010,1.25,1.00,0.409553,...,0.409553,0.415770,0.006217,0.636550,0.425873,0.510669,0.320872,0.320872,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...
4,time_based,yolo11m_yolov10n,yolo11m,yolov10n,123,0.55,0.010,1.00,1.00,0.414603,...,0.414603,0.415339,0.000736,0.644357,0.428201,0.510620,0.320059,0.320059,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...
5,time_based,yolo11m_yolov10n,yolo11m,yolov10n,2026,0.55,0.010,1.25,1.00,0.413925,...,0.413925,0.418851,0.004926,0.642293,0.429188,0.518211,0.319492,0.319492,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...
6,source_based,rtdetr_l_yolov10n,rtdetr_l,yolov10n,42,0.55,0.010,1.00,1.25,0.247704,...,0.247704,0.206570,-0.041134,0.378751,0.195745,0.219299,0.193840,0.193840,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...
7,source_based,rtdetr_l_yolov10n,rtdetr_l,yolov10n,123,0.55,0.001,1.00,1.25,0.285934,...,0.285934,0.196712,-0.089222,0.361614,0.185720,0.213602,0.179823,0.179823,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...
8,source_based,rtdetr_l_yolov10n,rtdetr_l,yolov10n,2026,0.55,0.010,1.00,1.25,0.302263,...,0.302263,0.209836,-0.092427,0.380554,0.200098,0.237304,0.182368,0.182368,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...
9,source_based,yolo11m_yolov10n,yolo11m,yolov10n,42,0.50,0.010,1.00,1.25,0.197783,...,0.197783,0.200247,0.002464,0.375060,0.188913,0.228266,0.172229,0.172229,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...


,protocol,pair,n_seeds,wbf_map50_95_mean,gain_over_best_member_mean,gain_over_best_member_sd,positive_gain_seed_count
0,source_based,rtdetr_l_yolov10n,3,0.204373,-0.074261,0.028733,0
1,source_based,yolo11m_yolov10n,3,0.207307,-0.004239,0.006480,1
2,time_based,rtdetr_l_yolov10n,3,0.368660,-0.073957,0.023076,0
3,time_based,yolo11m_yolov10n,3,0.416654,0.003960,0.002865,3


""


{
  "expected_cases": 12,
  "completed_cases": 12,
  "pending_cases": 0,
  "output_report": "/content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish/dataset_v2/experiments/multiseed_wbf_results_ready_v3_extension_final_combos/reports/new_wbf_gains.csv",
  "test_was_used_for_tuning": false,
  "ready_when_all_inputs_exist": true
}
